# Prompt-Variants + Ollama V2 -Notebook


In [1]:
'''Objective: 
Develop a small prototype (or Jupyter Notebook) that takes sample resume bullet points 
(provided below) and rewrites each one into improved bullet points using 3-4 different 
LLMs or different prompting techniques.  '''

'Objective: \nDevelop a small prototype (or Jupyter Notebook) that takes sample resume bullet points \n(provided below) and rewrites each one into improved bullet points using 3-4 different \nLLMs or different prompting techniques.  '

In [2]:
# Below code will have option to use both :- Different Prompting Techniques and Different LLM Models.
# Since I donot have enough money to invest on OpenAI, Anthropic and Gemini APIs. And free APIs of them have very limited number of tokens.
# I have used open source LLMs from OLLAMA
# This version of code will take input directly from user by pasting all bullet points. And pressing double enter button to execute. 
# Apart from that you can also enter bullet points manually line by line. And pressing double enter button to execute.

In [4]:
# Make sure you have installed Ollama and all relevant models used in below code.
# After that type "ollama serve"  in power shell so that it can run locally in computer.
# Next type ".venv/Scripts/activate" in powershell after correctly locating the folder in powershell.
# Next type "python -m pip install -U jupyterlab" in powershell to install jupyter notebook in folder
# To run jupyter notebook type "python -m jupyter lab" in powershell.   

In [5]:
# Notebook magic/shell command executed by Jupyter. Installs charset detection lib to reduce RequestsDependencyWarning.

In [6]:
! pip install charset-normalizer 

In [7]:
# Notebook magic/shell command executed by Jupyter. Installs the Python "ollama" package (note: not required if you use OpenAI client to hit Ollama’s OpenAI-compatible API)

In [8]:
! pip install ollama  

In [10]:
# imports  # Section header for imports.

import re  # Regex utilities for splitting/cleaning bullet text.
import os  # OS helpers (paths/env); not used yet but commonly kept for future use.
import requests  # HTTP client used to ping the Ollama server.
from openai import OpenAI  # OpenAI SDK client (works with Ollama's OpenAI-compatible endpoint).
from dotenv import load_dotenv  # Loads env vars from .env (used when you have api key).
from IPython.display import Markdown,display,update_display  # Notebook rendering helpers.

In [11]:
requests.get("http://localhost:11434").content  # Ping Ollama (server root) to confirm it is reachable; returns raw response bytes.

b'Ollama is running'

In [12]:
# constants  # Section header for configuration constants.
# MODEL_GPT = 'gpt-4o-mini'  # Example OpenAI model name (commented out / not used here).

OLLAMA_BASE_URL = "http://localhost:11434/v1"  # Base URL for Ollama's OpenAI-compatible API (must end with /v1).

MODEL_GPTOSS = 'gpt-oss:120b-cloud'  # Model tag (as available in your Ollama setup) for GPT OSS: 120b- cloud.
MODEL_GEMMA4 = 'gemma3:4b'  # Model tag for Gemma 3 4B.
MODEL_GEMMA = 'gemma3:1b'  # Model tag for Gemma 3 1B (fast/small).
MODEL_DEEPSEEK = 'deepseek-r1:1.5b'  # Model tag for DeepSeek R1 1.5B.
MODEL_LLAMA = 'llama3.2'  # Model tag for Llama 3.2.

ollama = OpenAI(base_url = OLLAMA_BASE_URL, api_key = "ollama")  # Create OpenAI client pointing to Ollama; api_key is a dummy placeholder.

In [13]:
# (Optional) Hardcoded bullets for testing (currently disabled).
#question = """  
#• Integrated the Google Form Transaction page with the website which lead to 2 min payment process, from 5min  # Example bullet 1 (commented out).
#• Designed and deployed the marketing web page section of the site.  # Example bullet 2 (commented out).
#• Developed and organized the Github repository to streamline workflow.  # Example bullet 3 (commented out).
#"""  
# End of commented-out sample input.

In [14]:
def get_question_from_user():  # Collects multiline bullet text from the user and normalizes it into clean "• ..." lines.
    print("Paste your experience (press Enter twice to finish):\n")  # Prompt user for input; double Enter ends entry.

    raw = []  # Accumulator list to store each non-empty line of user input.
    while True:  # Loop until the user enters a blank line.
        line = input()  # Read one line from stdin (Jupyter will show an input box).
        if line.strip() == "":  # Stop when the line is empty/whitespace-only.
            break  # Exit input loop.
        raw.append(line.rstrip())  # Store the line without trailing whitespace/newlines.

    text = " ".join(raw).strip()  # Combine lines into a single string and trim leading/trailing spaces.

    # Split on bullet symbols that appear anywhere (Word/PDF bullet: "", normal bullet: "•")  # Explains the regex split below.
    parts = re.split(r"\s*[•]\s+", text)  # Split text into bullet parts on either "•" or "" with flexible whitespace.

    # Remove junk/empties and clean extra spaces  # Explains cleanup step below.
    clean = [re.sub(r"\s+", " ", p).strip() for p in parts if p.strip()]  # Normalize internal whitespace + drop empty segments.

    question = "\n" + "\n".join(f"• {p}" for p in clean) + "\n"  # Rebuild as newline-separated bullets (each starting with "• ").
    return question  # Return the formatted bullet text for prompting.


question = get_question_from_user()  # Ask the user for bullets and store normalized result.
#display(Markdown(question))  # (Optional) Render the normalized bullets in Markdown (currently disabled).
print(question)  # Print the normalized bullets so you can verify formatting.

Paste your experience (press Enter twice to finish):



 • Scheduled appointments, managed emails and calls for senior leadership.   • Provided end-to-end event logistics and meeting support.   • Familiarity with audit techniques, processes, and tools.  • Proficiency in using GST portals for registration, payment, and return filing.  • Extracted, cleaned & integrated multi-source datasets (Insurance Records, Customer  Feedback); performed data profiling & executed sentiment analysis via Power Query to  deliver customer-centric insights.  • Designed & deployed dynamic dashboards using Bar, Line, Ribbon, Donut & Matrix  visualizations; implemented Role-Based Security (RLS) & scheduled automated data  refresh for real-time, secure reporting. 
 



• Scheduled appointments, managed emails and calls for senior leadership.
• Provided end-to-end event logistics and meeting support.
• Familiarity with audit techniques, processes, and tools.
• Proficiency in using GST portals for registration, payment, and return filing.
• Extracted, cleaned & integrated multi-source datasets (Insurance Records, Customer Feedback); performed data profiling & executed sentiment analysis via Power Query to deliver customer-centric insights.
• Designed & deployed dynamic dashboards using Bar, Line, Ribbon, Donut & Matrix visualizations; implemented Role-Based Security (RLS) & scheduled automated data refresh for real-time, secure reporting.



In [16]:
# System instruction: tells the model how to rewrite bullets.
system_prompt = """  
• Start  with a strong action verb (e.g., Analyzed, Led, Designed, Developed, etc.).  
• Include relevant industry keywords (e.g. for sales, marketing, software engineering).
• Use numerical metrics only when they can be directly inferred from the input; otherwise keep statements qualitative. 
• Be concise but informative (aim for ~12–20 words).  
• Maintain professional and clear language.  
• Make sentences more impact driven. And avoid generic fluff.  
• Make bullet points ATS friendly.  
"""  # End of system prompt string.


# Build the user prompt by appending the cleaned bullets.
user_prompt = "For each input bullet point, generate equal number of bullet points:" + question  

In [17]:
def get_prompt_versions(user_prompt, system_prompt):  # Builds different prompt styles (zero-shot, few-shot, etc.).

    zero_shot = [  # Zero-shot: system rules + raw user prompt.
        {"role": "system", "content": system_prompt},  # Provide the system instructions.
        {"role": "user", "content": user_prompt}  # Provide the actual bullet content/task.
    ]  # End zero-shot list.

    few_shot = [  # Few-shot: add one example pair before the real input.
        {"role": "system", "content": system_prompt},  # Provide the same system instructions.
        {"role": "user", "content":  # Provide an example in a user message.
         "Input: •Managed team meetings.\n"  # Example input bullet.
         "Output: •Led team coordination improving meeting efficiency by 30%.\n"},  # Example output bullet.
        {"role": "user", "content": user_prompt}  # Provide the real prompt after the example.
    ]  # End few-shot list.

    chain_of_thought = [  # CoT: instructs the model to reason step-by-step (may hurt small models sometimes).
        {"role": "system", "content": system_prompt},  # System rules.
        {"role": "user", "content":  # User message with CoT instruction.
         "Think step by step and rewrite professionally:\n\n" + user_prompt}  # CoT prefix + actual prompt.
    ]  # End CoT list.

    role_play = [  # Role-play: set a persona to influence tone.
        {"role": "system", "content":  # Persona system message.
         "You are a senior technical recruiter."},  # Recruiter persona.
        {"role": "system", "content": system_prompt},  # Formatting rules.
        {"role": "user", "content": user_prompt}  # Actual user prompt.
    ]  # End role-play list.

    return {  # Return dictionary of prompt variants for easy selection.
        "Zero-Shot": zero_shot,  # Key -> message list.
        "Few-Shot": few_shot,  # Key -> message list.
        "Chain-of-Thought": chain_of_thought,  # Key -> message list.
        "Role-Play (Recruiter)": role_play  # Key -> message list.
    }  # End return dict.


# ---------- Pretty Printer (for clarity) ----------  # Section divider for printing helpers.
def pretty_print_messages(messages):  # Prints each message so you can verify roles/contents.
    for i, msg in enumerate(messages, start=1):  # Iterate messages with 1-based index.
        print(f"\n--- MESSAGE {i} ---")  # Print message index header.
        print("ROLE   :", msg["role"])  # Print the role (system/user/assistant).
        print("CONTENT:")  # Print label for content.
        print(msg["content"])  # Print full content text.

In [18]:
prompts = get_prompt_versions(user_prompt, system_prompt)  # Build all prompt variants from your system + user prompt.

# Print all clearly  # Comment for the debug-print loop below.
for name, messages in prompts.items():  # Iterate each prompt variant (name -> message list).
    print(f"\n================ {name.upper()} ================")  # Print a header so variants are separated.
    pretty_print_messages(messages)  # Print each message in that variant for inspection.


================ ZERO-SHOT ================

--- MESSAGE 1 ---
ROLE   : system
CONTENT:
  
• Start  with a strong action verb (e.g., Analyzed, Led, Designed, Developed, etc.).  
• Include relevant industry keywords (e.g. for sales, marketing, software engineering).
• Use numerical metrics only when they can be directly inferred from the input; otherwise keep statements qualitative. 
• Be concise but informative (aim for ~12–20 words).  
• Maintain professional and clear language.  
• Make sentences more impact driven. And avoid generic fluff.  
• Make bullet points ATS friendly.  


--- MESSAGE 2 ---
ROLE   : user
CONTENT:
For each input bullet point, generate equal number of bullet points:
• Scheduled appointments, managed emails and calls for senior leadership.
• Provided end-to-end event logistics and meeting support.
• Familiarity with audit techniques, processes, and tools.
• Proficiency in using GST portals for registration, payment, and return filing.
• Extracted, cleaned & int

In [19]:
 # Old manual message construction (kept for reference, currently disabled).

#messages = [ 
#    {"role": "system", "content": system_prompt},  # System instructions.
#    {"role": "system", "content": user_prompt}  # should be role="user" if you re-enable this.

#]  # End commented-out messages list.

In [20]:
def run_chat(model, messages):  # Runs one chat completion call and displays Markdown output.
    response = ollama.chat.completions.create(  # Call Ollama's OpenAI-compatible chat completions endpoint.
        model=model,  # Select which local model to use.
        messages=messages  # Provide the message list (system/user) for the prompt variant.
    )  # End API call.

    reply = response.choices[0].message.content  # Extract the assistant's text from the first returned choice.

    # Fix common bullet formatting issue  # Explain the formatting normalization below.
    reply = reply.replace(" •", "\n•").replace("- ", "\n- ")  # Force bullets onto new lines for cleaner Markdown rendering.

    return display(Markdown(reply))  # Render the reply as Markdown output in the notebook.

In [33]:
run_chat(MODEL_GPTOSS, prompts["Chain-of-Thought"])  # Run the GPT-OSS model using the Chain-of-Thought prompt variant.


- **Coordinated** senior leadership calendars, streamlined email correspondence, and optimized inbound/outbound call management.  

- **Managed** full‑cycle event logistics and comprehensive meeting support, ensuring seamless execution and stakeholder satisfaction.  

- **Applied** audit methodologies, processes, and specialized tools to assess compliance and mitigate operational risk.  

- **Utilized** GST portals for entity registration, tax payment processing, and timely return filing.  

- **Extracted**, cleansed, and merged insurance and feedback datasets; performed data profiling and Power Query sentiment analysis delivering actionable customer insights.  

- **Designed** dynamic dashboards with varied visualizations, enforced role‑based security, and scheduled automated refresh for real‑time, secure reporting.

In [32]:
run_chat(MODEL_GEMMA4, prompts["Zero-Shot"])  # Run the gemma3 (4 billion parameters) model using the Zero-Shot prompt variant.

Okay, here are bullet points generated from your input, following your guidelines – aiming for approximately 12-20 words each, ATS-friendly, and focusing on impact:

*   **Supported** senior leadership through strategic appointment scheduling and effective communication management.
*   **Orchestrated** seamless event logistics and comprehensive meeting support, ensuring optimal execution.
*   **Applied** deep auditing expertise to analyze processes and leverage relevant tools effectively.
*   **Streamlined** GST compliance by registering, processing payments, and filing returns accurately.
*   **Transformed** raw data into actionable insights via Power Query, delivering customer-focused analytics.
*   **Developed** interactive dashboards with diverse visualizations optimizing data representation & securing access. 

**Do you want me to generate additional bullet points, or would you like me to refine these based on a specific job description?**

In [31]:
run_chat(MODEL_LLAMA, prompts["Few-Shot"])  # Run the llama3.2  model using the Few-Shot prompt variant.

Here are the generated bullet points:

**Scheduled appointments, managed emails and calls for senior leadership.**
• Coordinated high-priority tasks with stakeholders to meet deadlines and drive results.

**Provided end-to-end event logistics and meeting support.**
• Managed seamless execution of events, ensuring timely setup, and providing exceptional client experiences.

**Familiarity with audit techniques, processes, and tools.**
• Demonstrated expertise in conducting thorough audits using industry-standard methodologies for robust assurance.

**Proficiency in using GST portals for registration, payment, and return filing.**
• Ensured accurate and compliant submission of GST returns and payments on behalf of clients amidst regulatory complexities.

**Extracted, cleaned & integrated multi-source datasets (Insurance Records, Customer Feedback); performed data profiling & executed sentiment analysis via Power Query to deliver customer-centric insights.**
• Analyzed diverse dataset elements using advanced Microsoft Power Query techniques for actionable customer insights driving business value.

**Designed & deployed dynamic dashboards using Bar, Line, Ribbon, Donut & Matrix visualizations; implemented Role-Based Security (RLS) & scheduled automated data refresh for real-time, secure reporting.**
• Built intuitive and interactive dashboards leveraging cutting-edge visualization tools for high visibility of key performance metrics among designated user groups.

In [47]:
run_chat(MODEL_DEEPSEEK, prompts["Role-Play (Recruiter)"])  # Run the deepseek r1  model using the Role-Play(Recruiter) prompt variant.

1. Scheduling Senior Leadership meetings, managing email notifications, and tracking calls with automated response systems for reports.  
2. Completing audit plans, cleaning internal datasets, and ensuring accurate progress reporting through documentation reviews.  
3. Setting up GST portals accurately, understanding billing processes, and implementing data integrity solutions through auditing and testing.  
4. Choosing the right integration tools, conducting thorough performance tests, and planning for AI/ML adoption based on business needs.  
5. Deploying dynamic dashboards with specific visualizations using Bar, Line, Ribbon, Donut, and Matrix visualizations in meetings.  
6. Ensuring real-time reporting, secure data handling through role-based access control (RLS), and implementing security measures for compliance purposes.

In [49]:
run_chat(MODEL_GEMMA, prompts["Few-Shot"])  # Run the Model_GEMMA model using the Few-Shot prompt variant.

Okay, here’s a list of six bullet points, each consistent with your requested format and keywords, aiming for impact and conciseness:

*   **Developed & executed strategic initiatives to enhance sales performance, targeting key market segments.**
*   **Streamlined communication workflows, scheduling appointments and calls for executive leadership.**
*   **Deeply understood audit methodologies, processes, and tools to ensure compliance and accuracy.**
*   **Expertly utilized GST portals for seamless registration, payment & return filings, minimizing delays.**
*   **Analyzed and transformed complex data sets – Insurance Records, Customer Feedback – into actionable insights.**
*   **Crafted dynamic dashboards with advanced visualizations (Bar, Line, Ribbon, Donut, Matrix) for real-time reporting & customer understanding.**

---

**Notes on ATS Compatibility:**

*   I’ve focused on using strong verbs and concise language, which is generally well-received by ATS systems.
*   The bullet points are formatted with clear separation using commas and spacing. 
*   I’ve avoided phrases like "responsible for" or "involved in" which can sometimes trigger ATS parsing issues.

Would you like me to refine any of these further or create more options?

In [38]:
# (empty cell)  # This cell intentionally has no executable code.

In [ ]:
"""
 GPT-OSS:120b and gemma3:4b performs relatively good as an open source model.
 llama 3.2 has relatively below average performance as compare to other.
 Prompting Technique:- Adopted "policy style" approach for prompting.
                       Used very simple words so that open source light LLMs can easily interpret.
                       Avoided using fake numerical values in the output of LLM models.
"""